# Transcoders: Bridging Nonlinear Layers

**Survey Reference:** Section 7.4.1 - Transcoders: Bridging Nonlinear Layers

## Overview

**Transcoders** learn sparse feature representations that map from MLP inputs to MLP outputs, effectively learning the computation performed by the MLP in terms of interpretable features.

### Why Transcoders?

Standard SAEs on MLP activations miss the computational transformation:
- SAE on MLP input: Features describe what goes in
- SAE on MLP output: Features describe what comes out  
- **Transcoder:** Features describe the transformation itself

## Learning Objectives

1. Understand the transcoder architecture
2. Compare transcoders vs. standard SAEs
3. Train transcoders on MLP layers
4. Interpret transcoder features as input→output mappings

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm

device = "cuda" if torch.cuda.is_available() else "cpu"

## 1. The Problem with Standard SAEs for MLPs

Consider a transformer MLP:

$$\text{MLP}(x) = W_{out} \cdot \sigma(W_{in} \cdot x + b_{in}) + b_{out}$$

A standard SAE learns:
- On input $x$: Features present before transformation
- On output $\text{MLP}(x)$: Features present after transformation

But neither captures: **"This input feature causes this output feature"**

## 2. Transcoder Architecture

A transcoder maps from MLP input to MLP output:

$$f = \text{ReLU}(W_e \cdot x + b_e)$$
$$\hat{y} = W_d \cdot f + b_d$$

Where:
- $x$ is the MLP input (residual stream)
- $y$ is the MLP output  
- $f$ is the sparse feature representation
- The transcoder learns to approximate $y \approx \hat{y}$

In [ ]:
class Transcoder(nn.Module):
    """
    Transcoder: Maps MLP input -> sparse features -> MLP output.
    
    Unlike SAEs, transcoders model the transformation, not just the representation.
    """
    
    def __init__(self, d_model: int, n_features: int):
        super().__init__()
        self.d_model = d_model
        self.n_features = n_features
        
        # Encoder: map input to sparse features
        self.W_enc = nn.Parameter(torch.randn(d_model, n_features) / np.sqrt(d_model))
        self.b_enc = nn.Parameter(torch.zeros(n_features))
        
        # Decoder: map features to output space
        self.W_dec = nn.Parameter(torch.randn(n_features, d_model) / np.sqrt(n_features))
        self.b_dec = nn.Parameter(torch.zeros(d_model))
    
    def encode(self, x: torch.Tensor) -> torch.Tensor:
        """Encode MLP input to sparse features."""
        pre_acts = x @ self.W_enc + self.b_enc
        return F.relu(pre_acts)
    
    def decode(self, f: torch.Tensor) -> torch.Tensor:
        """Decode features to MLP output space."""
        return f @ self.W_dec + self.b_dec
    
    def forward(self, x: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        """
        Forward pass.
        
        Args:
            x: MLP input [batch, d_model]
            
        Returns:
            y_hat: Predicted MLP output [batch, d_model]
            f: Sparse feature activations [batch, n_features]
        """
        f = self.encode(x)
        y_hat = self.decode(f)
        return y_hat, f
    
    def get_feature_effect(self, feature_idx: int) -> torch.Tensor:
        """
        Get the effect of a feature on the output.
        
        Returns the decoder column for this feature (what it adds to output).
        """
        return self.W_dec[feature_idx]

## 3. Transcoder Loss Function

The transcoder loss predicts the MLP output (not reconstruction of input):

$$\mathcal{L} = \|y - \hat{y}\|_2^2 + \lambda \|f\|_1$$

In [ ]:
def transcoder_loss(y_true: torch.Tensor, y_pred: torch.Tensor, 
                    f: torch.Tensor, l1_coef: float = 1e-3) -> tuple[torch.Tensor, dict]:
    """
    Compute transcoder loss.
    
    Args:
        y_true: Actual MLP output
        y_pred: Predicted MLP output from transcoder
        f: Feature activations
        l1_coef: Sparsity coefficient
    """
    # Prediction loss
    pred_loss = (y_true - y_pred).pow(2).mean()
    
    # Sparsity loss
    l1_loss = f.abs().mean()
    
    total_loss = pred_loss + l1_coef * l1_loss
    
    with torch.no_grad():
        # Fraction of variance explained
        fve = 1 - (y_true - y_pred).var() / y_true.var()
        # L0 (average active features)
        l0 = (f > 0).float().sum(dim=1).mean()
    
    metrics = {
        'loss': total_loss.item(),
        'pred_loss': pred_loss.item(),
        'l1_loss': l1_loss.item(),
        'fve': fve.item(),
        'l0': l0.item(),
    }
    
    return total_loss, metrics

## 4. Synthetic Experiment: Learning MLP Computation

Let's create a synthetic MLP and train a transcoder to model it.

In [ ]:
class SyntheticMLP(nn.Module):
    """A simple MLP layer for testing."""
    
    def __init__(self, d_model: int, d_hidden: int):
        super().__init__()
        self.W_in = nn.Parameter(torch.randn(d_model, d_hidden) / np.sqrt(d_model))
        self.W_out = nn.Parameter(torch.randn(d_hidden, d_model) / np.sqrt(d_hidden))
        self.b_in = nn.Parameter(torch.zeros(d_hidden))
        self.b_out = nn.Parameter(torch.zeros(d_model))
    
    def forward(self, x):
        h = F.relu(x @ self.W_in + self.b_in)
        return h @ self.W_out + self.b_out

# Create synthetic MLP
d_model = 64
d_hidden = 256

mlp = SyntheticMLP(d_model, d_hidden)

# Generate input/output pairs
n_samples = 10000
X = torch.randn(n_samples, d_model)
with torch.no_grad():
    Y = mlp(X)

print(f"Input shape: {X.shape}")
print(f"Output shape: {Y.shape}")

In [ ]:
def train_transcoder(transcoder, X, Y, n_epochs=100, batch_size=256, 
                     lr=1e-3, l1_coef=1e-3):
    """Train a transcoder to model MLP computation."""
    optimizer = torch.optim.Adam(transcoder.parameters(), lr=lr)
    history = []
    
    for epoch in tqdm(range(n_epochs)):
        perm = torch.randperm(X.size(0))
        epoch_metrics = []
        
        for i in range(0, X.size(0), batch_size):
            x_batch = X[perm[i:i+batch_size]].to(device)
            y_batch = Y[perm[i:i+batch_size]].to(device)
            
            optimizer.zero_grad()
            y_pred, f = transcoder(x_batch)
            loss, metrics = transcoder_loss(y_batch, y_pred, f, l1_coef)
            loss.backward()
            optimizer.step()
            
            epoch_metrics.append(metrics)
        
        avg_metrics = {k: np.mean([m[k] for m in epoch_metrics]) for k in epoch_metrics[0]}
        history.append(avg_metrics)
        
        if epoch % 20 == 0:
            print(f"Epoch {epoch}: Loss={avg_metrics['loss']:.4f}, FVE={avg_metrics['fve']:.4f}, L0={avg_metrics['l0']:.1f}")
    
    return history

# Train transcoder
transcoder = Transcoder(d_model, n_features=512).to(device)
history = train_transcoder(transcoder, X, Y, n_epochs=100, l1_coef=1e-3)

## 5. Interpreting Transcoder Features

Each transcoder feature represents:
- **Encoder column:** What input pattern activates this feature
- **Decoder column:** What output pattern this feature produces

In [ ]:
def analyze_transcoder_feature(transcoder, feature_idx):
    """Analyze what a transcoder feature computes."""
    with torch.no_grad():
        # Input pattern that activates this feature
        input_pattern = transcoder.W_enc[:, feature_idx].cpu().numpy()
        
        # Output pattern this feature produces
        output_pattern = transcoder.W_dec[feature_idx].cpu().numpy()
    
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    
    axes[0].bar(range(len(input_pattern)), input_pattern)
    axes[0].set_title(f"Feature {feature_idx}: Input Pattern")
    axes[0].set_xlabel("Dimension")
    
    axes[1].bar(range(len(output_pattern)), output_pattern)
    axes[1].set_title(f"Feature {feature_idx}: Output Pattern")
    axes[1].set_xlabel("Dimension")
    
    plt.tight_layout()
    plt.show()

# TODO: Analyze specific features
# analyze_transcoder_feature(transcoder, 0)

## 6. Comparison: SAE vs. Transcoder

| Aspect | SAE | Transcoder |
|--------|-----|------------|
| Input | Activations | MLP input |
| Output | Same activations | MLP output |
| Features represent | What's present | What computation happens |
| Use case | Understanding representations | Understanding transformations |

In [ ]:
# TODO: Compare SAE features on MLP output vs transcoder features

## Key Takeaways

1. **Transcoders model computation,** not just representation
2. **Encoder columns** show what inputs activate each feature
3. **Decoder columns** show what each feature adds to output
4. **Better for MLP analysis** than standard SAEs
5. **Features are causal:** Activating a feature → specific output

## Next Steps

→ **03_crosscoders.ipynb**: Analyze features that span multiple layers